# Sonar Mind: The Educational Assistant

Capstone notebook for a lecture assistant that transcribes audio, answers questions from that lecture only, and builds a quiz, flashcards, and a mind map.

**Team (VIT-AP, B.Tech CSBS, August 2025)**
- Patan Mohammed Ibrahim Khan (22BCB7267), team lead
- Duggireddy Venu (22BCB7275)
- Maile Hruday Raj (22BCB7132)
- Sanikommu Divakar Reddy (22BCB7003)
- Guide: Dr. Hussain Syed

The implementation lives in the `sonar_mind` package so this notebook and the Hugging Face app (`app.py`) stay on the same code. Run the cells from top to bottom. The last cell opens the interface.

Create a `.env` file with `GOOGLE_API_KEY` before transcription, indexing, quizzes, flashcards, or mind maps. The checks in this notebook do not call the API.


## Why this version runs

The August 2025 report installed `gradio==4.14.0`, `google-generativeai==0.3.2`, and `fastapi==0.112.2`, and it called `gemini-2.0-flash` and `text-embedding-004`. Those models are retired, those pins conflict, the retrieval chain stopped before it was returned, and one error string was missing a quote. This notebook uses the repaired package and current Gemini model names from `.env` (`GEMINI_MODEL`, `GEMINI_EMBED_MODEL`).


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "app.py").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sonar_mind.config import GEMINI_EMBED_MODEL, GEMINI_MODEL, get_api_key

print("Project:", ROOT)
print("Chat model:", GEMINI_MODEL)
print("Embedding model:", GEMINI_EMBED_MODEL)
try:
    get_api_key()
    print("GOOGLE_API_KEY is set.")
except RuntimeError as exc:
    print(exc)


## 1. Lecture text

Teachers upload audio in the app. This cell loads the included sample lecture so the rest of the notebook can be inspected without a recording. `load_pasted_transcript` is the same function the Paste button uses.


In [ ]:
from sonar_mind.rag import chunk_transcript
from sonar_mind.transcription import load_pasted_transcript

lecture = (ROOT / "sample_data" / "sample_lecture.txt").read_text(encoding="utf-8")
full_text, status = load_pasted_transcript(lecture)
chunks = chunk_transcript(full_text, chunk_size=500, overlap_size=80)
print(status)
print(f"Characters: {len(full_text)} | Chunks: {len(chunks)}")
print("--- first chunk ---")
print(chunks[0].page_content[:500])


## 2. Transcription

`transcribe_audio_chunk` uploads a file with the Gemini Files API and asks for plain text plus key points. The Gradio handler appends each successful chunk and clears the player. A missing file returns a status message instead of raising.


In [ ]:
from sonar_mind.transcription import handle_transcription_request

latest, full, _audio, status, input_status = handle_transcription_request(None)
print(status)
print(input_status)
print("Stored transcript still has the sample:", "foreign key" in full.lower())


## 3. Index and grounded questions

`handle_indexing_request` splits the transcript, embeds the chunks, stores them in Chroma, and builds a retrieval chain. `handle_query_request` answers from those chunks. If you have not indexed yet, it tells you to index first instead of calling the model with an empty database.

Run the next cell only after `GOOGLE_API_KEY` is set. It spends a small amount of API quota.


In [ ]:
import os
from sonar_mind.rag import handle_indexing_request, handle_query_request
from sonar_mind.state import get_full_transcript_text

if not os.environ.get("GOOGLE_API_KEY"):
    print("Skipped live indexing. Add GOOGLE_API_KEY to .env and run this cell again.")
    print(handle_query_request("What is a foreign key?"))
else:
    print(handle_indexing_request(get_full_transcript_text()))
    print(handle_query_request("What is a foreign key?"))
    print(handle_query_request("What is the capital of France?"))


## 4. Quiz, flashcards, and mind map

Study tools ask Gemini for JSON, then the app scores the quiz and tracks flashcards locally. The cell below exercises that scoring with a fixed quiz so it passes with no API key. Empty transcripts are rejected before any model call.


In [ ]:
from tests.test_offline import (
    test_chunking_and_paste,
    test_flashcard_flow_without_model,
    test_flashcards_reject_empty,
    test_generate_quiz_rejects_empty,
    test_json_cleanup,
    test_mindmap_layout_has_no_script,
    test_quiz_flow_without_model,
)

test_json_cleanup()
test_chunking_and_paste()
test_quiz_flow_without_model()
test_flashcard_flow_without_model()
test_mindmap_layout_has_no_script()
test_generate_quiz_rejects_empty()
test_flashcards_reject_empty()
print("Offline notebook checks passed.")


## 5. Open the assistant

This cell builds the five-tab Gradio app: transcription, questions, quiz, flashcards, and mind map. Stop the cell to shut the app down. On Hugging Face Spaces the same UI is `app.py`; put the key in a Space secret named `GOOGLE_API_KEY`.


In [ ]:
from app import demo

demo.launch()
